# Deploy Prep

Upload the pixels wheel and OHIF archive to the UC Volume,
and rewrite each app's `requirements.txt` to reference the Volume wheel.

In [0]:
import os
_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_repo_root = os.path.normpath(os.path.join("/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get()), ".."))

%pip install --quiet -r {_repo_root}/requirements.txt
dbutils.library.restartPython()

In [0]:
%run ./config/proxy_prep

In [0]:
sql_warehouse_id, table, volume = init_widgets(show_volume=True)
init_env()

# Upload Wheel and OHIF Archive to Volume

In [ ]:
import shutil, os, glob, tarfile, re

# Compute repo root from notebook path
_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_dir = "/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get())
_repo_root = os.path.normpath(os.path.join(_nb_dir, ".."))

_dicomweb_gateway_path = os.path.join(_repo_root, "apps", "dicom-web-gateway")
_dicomweb_path = os.path.join(_repo_root, "apps", "dicom-web")
_view_app_path = os.path.join(_repo_root, "apps", "view-app")

# Derive volume paths from the volume widget (catalog.schema.volume_name)
_vol_parts = volume.split(".")
_volume_base = f"/Volumes/{_vol_parts[0]}/{_vol_parts[1]}/{_vol_parts[2]}"

# 1. Upload lightweight wheel to UC Volume
_wheel_pattern = os.path.join(_repo_root, "dist", "databricks_pixels-*.whl")
_wheels = sorted(glob.glob(_wheel_pattern))
assert _wheels, f"No wheel found at {_wheel_pattern} — run 'make build' before deploying"
_wheel_path = _wheels[-1]
_wheel_name = os.path.basename(_wheel_path)
_vol_dist_dir = os.path.join(_volume_base, "dist")
os.makedirs(_vol_dist_dir, exist_ok=True)
_vol_wheel_path = os.path.join(_vol_dist_dir, _wheel_name)
shutil.copy2(_wheel_path, _vol_wheel_path)
print(f"Uploaded {_wheel_name} ({os.path.getsize(_wheel_path)/1e6:.1f}MB) to {_vol_wheel_path}")

# 2. Upload OHIF archive to Volume (prefer prebuilt dist/ohif.tar.gz from sync)
_ohif_dist = os.path.join(_repo_root, "dist", "ohif.tar.gz")
_ohif_src = os.path.join(_repo_root, "apps", "dicom-web", "ohif")
_ohif_archive = os.path.join(_volume_base, "ohif.tar.gz")
if os.path.isfile(_ohif_dist):
    shutil.copy2(_ohif_dist, _ohif_archive)
    print(f"Uploaded OHIF archive ({os.path.getsize(_ohif_archive)/1e6:.1f}MB) from dist to {_ohif_archive}")
elif os.path.isdir(_ohif_src) and len(os.listdir(_ohif_src)) > 5:
    with tarfile.open(_ohif_archive, "w:gz") as tar:
        tar.add(_ohif_src, arcname="ohif")
    print(f"Uploaded OHIF archive ({os.path.getsize(_ohif_archive)/1e6:.1f}MB) from src to {_ohif_archive}")
elif os.path.isfile(_ohif_archive):
    print(f"OHIF archive already on Volume ({os.path.getsize(_ohif_archive)/1e6:.1f}MB)")
else:
    raise FileNotFoundError(
        f"OHIF archive not found at dist={_ohif_dist}, src={_ohif_src}, or vol={_ohif_archive} — run 'make build' first"
    )

# Rewrite App requirements.txt

In [ ]:
# 3. Clean up leftover .whl files in app dirs (from prior runs)
for _app_dir in [_dicomweb_path, _dicomweb_gateway_path, _view_app_path]:
    for _old_whl in glob.glob(os.path.join(_app_dir, "*.whl")):
        os.remove(_old_whl)

# 4. Rewrite each app's requirements.txt to reference the Volume wheel.
_wheel_version = re.search(r"databricks_pixels-([^-]+)-", _wheel_name)
_version_comment = f"# pixels-wheel: {_wheel_version.group(1)}" if _wheel_version else ""

for _app_dir in [_dicomweb_path, _dicomweb_gateway_path, _view_app_path]:
    _req_path = os.path.join(_app_dir, "requirements.txt")
    with open(_req_path, "r") as f:
        _lines = f.readlines()
    with open(_req_path, "w") as f:
        _lines = [l for l in _lines if not l.strip().startswith("# pixels-wheel:")]
        if _version_comment:
            f.write(_version_comment + "\n")
        for _line in _lines:
            if (_line.strip() == "databricks-pixels"
                or _line.strip().startswith("databricks-pixels==")
                or _line.strip().startswith("./databricks_pixels")
                or _line.strip().startswith("/Volumes/")):
                f.write(f"{_vol_wheel_path}\n")
            else:
                f.write(_line)
    print(f"  Updated {os.path.basename(_app_dir)}/requirements.txt -> {_wheel_name}")

print("Done: deploy prep complete")